# Khảo sát hiệu suất bằng các mô hình Học máy truyền thống (Machine Learning)

Notebook này chỉ dùng các **Tabular Data** của riêng từng tài khoản / ví thông qua Class `EllipticDataset` thay vì đưa vào biểu diễn GNN. Điều này giúp chúng ta dễ dàng so sánh một cách thực nghiệm.

Chúng ta sẽ thử nghiệm các mô hình cơ bản và nâng cao sau:
1. **Decision Tree**
2. **Random Forest** 
3. **K-Nearest Neighbors (KNN)**
4. **Naïve Bayes**
5. **Bagging**
6. **LightGBM**
7. **XGBoost**

In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from omegaconf import OmegaConf

# Machine Learning libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
import xgboost as xgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from fraud_detection import EllipticDataset

## 1. Chuẩn bị dữ liệu Tabular
Sử dụng `EllipticDataset` để có chung bộ ID và ground truth.

In [2]:
# Load một tham số config bất kỳ (gat/gcn đều được vì chỉ cần path về dataset)
config = OmegaConf.load("configs/elliptic_gat.yaml")
dataset = EllipticDataset(config.dataset)

# Chuyển Tensor về dạng Numpy Numpy 
node_features = dataset.node_features.numpy()
labels = dataset.labels.numpy()

# Sinh ID cho tập Train và Valid thông qua hàm của Dataset
train_idx, valid_idx = dataset.train_test_split()

X_train = node_features[train_idx]
y_train = labels[train_idx]

X_valid = node_features[valid_idx]
y_valid = labels[valid_idx]

print(f"🔹 [Thông tin dữ liệu]")
print(f"   - Tập huấn luyện (Train): {X_train.shape[0]} samples")
print(f"   - Tập đánh giá (Valid): {X_valid.shape[0]} samples")
print(f"   - Số lượng đặc trưng (Features): {X_train.shape[1]}")

🔹 [Thông tin dữ liệu]
   - Tập huấn luyện (Train): 29894 samples
   - Tập đánh giá (Valid): 16670 samples
   - Số lượng đặc trưng (Features): 165


## 2. Huấn luyện (Training) & Đánh giá (Evaluation)
Việc xử lý sẽ được chạy tuần tự qua các model dưới dạng Dictionary.

In [3]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42, n_jobs=-1),
    "KNN": KNeighborsClassifier(n_jobs=-1),
    "Naïve Bayes": GaussianNB(),
    "Bagging": BaggingClassifier(random_state=42, n_jobs=-1),
    "LightGBM": lgb.LGBMClassifier(random_state=42, n_jobs=-1, verbose=-1),
    "XGBoost": xgb.XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss')
}

results = []

print("Bắt đầu huấn luyện...\n" + "-"*40)

for name, model in models.items():
    print(f"🔄 Đang chạy: {name}...")
    
    # Huấn luyện mô hình
    model.fit(X_train, y_train)
    
    # Dự đoán
    y_pred = model.predict(X_valid)
    
    # Thu thập Evaluation Metrics
    acc = accuracy_score(y_valid, y_pred)
    prec = precision_score(y_valid, y_pred, zero_division=0)
    rec = recall_score(y_valid, y_pred, zero_division=0)
    f1 = f1_score(y_valid, y_pred, zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1-Score": f1
    })
    
print("-"*40 + "\n✅ Hoàn tất toàn bộ chu trình!")

Bắt đầu huấn luyện...
----------------------------------------
🔄 Đang chạy: Decision Tree...
🔄 Đang chạy: Random Forest...
🔄 Đang chạy: KNN...
🔄 Đang chạy: Naïve Bayes...
🔄 Đang chạy: Bagging...
🔄 Đang chạy: LightGBM...
🔄 Đang chạy: XGBoost...
----------------------------------------
✅ Hoàn tất toàn bộ chu trình!


## 3. Tổng hợp Báo cáo
Xem bảng kết quả dưới đây, mô hình nào có chỉ số `F1 (Macro)` hoặc `Accuracy` cao nhất sẽ được xem là tốt nhất.

In [4]:
df_results = pd.DataFrame(results)

# Sắp xếp bảng tính theo chiều giảm dần của F1
df_results = df_results.sort_values(by="F1-Score", ascending=False).reset_index(drop=True)

display(df_results)

,Model,Accuracy,Precision,Recall,F1-Score
0,Random Forest,0.980444,0.969021,0.722068,0.827513
1,LightGBM,0.978164,0.926453,0.721145,0.811007
2,XGBoost,0.977504,0.913551,0.722068,0.806601
3,Bagging,0.974925,0.875706,0.715605,0.787602
4,KNN,0.946311,0.576672,0.652816,0.612386
5,Decision Tree,0.922076,0.442184,0.762696,0.559810
6,Naïve Bayes,0.701020,0.163533,0.875346,0.275581
